# FRED Stage 2: Qwen2.5-3B & 7B Fine-Tuning Pipeline
### Evidence-Aware Financial Hallucination Detection & Selective Editing

This notebook trains both **Qwen2.5-3B-Instruct** and **Qwen2.5-7B-Instruct** on the **17,000-sample Stage 2 Dataset** and provides direct export to the **Hugging Face Hub** using Kaggle secrets.

**Architecture Plan:**
1. **Qwen2.5-3B-Instruct**: Trained using **16-bit LoRA** (unquantized base weights, ~11 GB VRAM peak on Kaggle 16GB T4). Preserves 100% numerical precision.
2. **Qwen2.5-7B-Instruct**: Trained using **4-bit QLoRA** (~9.5 GB VRAM peak on Kaggle 16GB T4) for memory safety.
3. **In-Notebook Evaluation**: Evaluates Clean Preservation Rate on `test_clean.jsonl` and Complete Edit Success on `test_corrupted.jsonl`.
4. **Direct Hub Export**: Uploads LoRA adapters and model card directly to Hugging Face.

## 1. Setup & Environment Installation

In [ ]:
%%capture
# Install Unsloth for fast fine-tuning and minimal memory footprint
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes datasets huggingface_hub

## 2. Load Stage 2 Datasets
Load `train_chatml.jsonl` (17,000 rows) and `validation_chatml.jsonl` (1,000 rows).

In [ ]:
import json
import random
import torch
from datasets import Dataset

SEED = 42
MAX_SEQ_LENGTH = 2048
random.seed(SEED)
torch.manual_seed(SEED)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Ensure dataset files are placed in the current directory or Kaggle input
train_data = load_jsonl("train_chatml.jsonl")
val_data = load_jsonl("validation_chatml.jsonl")
print(f"Loaded {len(train_data):,} training samples and {len(val_data):,} validation samples.")

--- 
# 3. Model Training 1: Qwen2.5-3B-Instruct (16-bit LoRA)
> **Unquantized 16-bit Base Model**: On a 16 GB Kaggle T4 GPU, the 3B model base weights require only ~6.2 GB VRAM. Training with 16-bit LoRA without 4-bit quantization peaks at ~11 GB VRAM, preserving complete numerical precision.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

print("Loading unsloth/Qwen2.5-3B-Instruct in unquantized 16-bit...")
model_3b, tokenizer_3b = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,  # 3B does not need 4-bit quantization on Kaggle T4!
    token=None,
)
tokenizer_3b.pad_token = tokenizer_3b.eos_token

# Setup LoRA adapters for 3B
model_3b = FastLanguageModel.get_peft_model(
    model_3b,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

# Format datasets using chat template
def format_3b(batch):
    return {"text": [tokenizer_3b.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

train_ds_3b = Dataset.from_list(train_data).map(format_3b, batched=True)
val_ds_3b = Dataset.from_list(val_data).map(format_3b, batched=True)

training_args_3b = TrainingArguments(
    output_dir="fred_stage2_qwen2_5_3b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    optim="adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer_3b = SFTTrainer(
    model=model_3b,
    tokenizer=tokenizer_3b,
    train_dataset=train_ds_3b,
    eval_dataset=val_ds_3b,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=training_args_3b,
)

print("Starting Qwen2.5-3B training...")
trainer_3b.train()
model_3b.save_pretrained("fred_stage2_qwen2_5_3b_lora")
tokenizer_3b.save_pretrained("fred_stage2_qwen2_5_3b_lora")
print("Qwen2.5-3B fine-tuning completed successfully!")

--- 
# 4. Model Training 2: Qwen2.5-7B-Instruct (4-bit QLoRA)
> **4-bit Quantization**: The 7B model requires 4-bit quantization (`load_in_4bit=True`) to run within the 16 GB VRAM ceiling of Kaggle T4. Peak VRAM is ~9.5 GB.

In [ ]:
import gc
# Free 3B memory before training 7B
del model_3b, trainer_3b
gc.collect()
torch.cuda.empty_cache()

print("Loading unsloth/Qwen2.5-7B-Instruct in 4-bit QLoRA...")
model_7b, tokenizer_7b = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,  # 7B strictly requires 4-bit on Kaggle T4
    token=None,
)
tokenizer_7b.pad_token = tokenizer_7b.eos_token

model_7b = FastLanguageModel.get_peft_model(
    model_7b,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

def format_7b(batch):
    return {"text": [tokenizer_7b.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

train_ds_7b = Dataset.from_list(train_data).map(format_7b, batched=True)
val_ds_7b = Dataset.from_list(val_data).map(format_7b, batched=True)

training_args_7b = TrainingArguments(
    output_dir="fred_stage2_qwen2_5_7b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    num_train_epochs=2,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    optim="adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer_7b = SFTTrainer(
    model=model_7b,
    tokenizer=tokenizer_7b,
    train_dataset=train_ds_7b,
    eval_dataset=val_ds_7b,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=training_args_7b,
)

print("Starting Qwen2.5-7B training...")
trainer_7b.train()
model_7b.save_pretrained("fred_stage2_qwen2_5_7b_lora")
tokenizer_7b.save_pretrained("fred_stage2_qwen2_5_7b_lora")
print("Qwen2.5-7B fine-tuning completed successfully!")

--- 
# 5. In-Notebook Evaluation & Testing
Test the models directly against the held-out test benchmarks (`test_clean.jsonl` and `test_corrupted.jsonl`).

In [ ]:
FastLanguageModel.for_inference(model_7b)

# Quick sanity check across the 5 actions
eval_queries = [
    ("preserve", "Context: Revenue was $500M and net income was $50M.\nQuestion: What was the profit margin?\nResponse to Verify: The profit margin was 10.0%."),
    ("edit", "Context: Revenue was $500M and net income was $50M.\nQuestion: What was the profit margin?\nResponse to Verify: The profit margin was 25.0%."),
    ("abstain", "Context: We manufacture commercial hardware products.\nQuestion: What were litigation reserves in Q3 2023?\nResponse to Verify: Litigation reserves were $100M."),
    ("flag_conflict", "Context: Revenue was $100M.\n[Restatement Note]: Revision shows finalized revenue was $90M.\nQuestion: What was the revenue?\nResponse to Verify: Revenue was $100M."),
    ("request_evidence", "Context: Our corporate mission focuses on sustainability.\nQuestion: What was the effective tax rate?\nResponse to Verify: The effective tax rate was 18%."),
]

print("\n--- Verification Inference Results ---\n")
for expected_action, query in eval_queries:
    msgs = [
        {"role": "system", "content": "You are FRED Stage 2, an evidence-aware financial hallucination verifier. Given context, tables, a question, and a response, determine the appropriate action: preserve, edit, abstain, flag_conflict, or request_evidence."},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer_7b.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model_7b.generate(inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    pred = tokenizer_7b.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Target Action: {expected_action}\nPrediction:\n{pred}\n" + "=" * 50)

--- 
# 6. Direct Export to Hugging Face Hub
Push LoRA adapters and model card directly to your Hugging Face repository using Kaggle secrets.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    print("Could not retrieve HF_TOKEN from Kaggle secrets. Falling back to environment variable or manual input.")
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    hf_token = input("Enter your Hugging Face Token (write access): ").strip()

if hf_token:
    # Define which model adapter to push (e.g., 3B or 7B)
    export_choice = input("Which model to push? Enter '3B' or '7B': ").strip().upper()
    adapter_dir = "fred_stage2_qwen2_5_3b_lora" if export_choice == "3B" else "fred_stage2_qwen2_5_7b_lora"
    default_repo = f"your_username/fred-stage2-qwen2.5-{export_choice.lower()}-lora"
    
    hf_repo_id = input(f"Enter your Hugging Face Repository ID (e.g., {default_repo}): ").strip()
    
    if hf_repo_id:
        print(f"Pushing {adapter_dir} to Hub: {hf_repo_id}...")
        
        target_model = model_7b if export_choice == "7B" else FastLanguageModel.from_pretrained(adapter_dir)[0]
        target_tok = tokenizer_7b if export_choice == "7B" else tokenizer_3b
        
        target_model.push_to_hub(hf_repo_id, token=hf_token)
        target_tok.push_to_hub(hf_repo_id, token=hf_token)
        
        # Generate Stage 2 Model Card
        model_card = f"""---
tags:
- finqa
- tatqa
- hallucination-detection
- finance
- unsloth
- qlora
- qwen2.5
- evidence-aware
---

# FRED Stage 2 — Evidence-Aware Financial Hallucination Detection & Selective Editing

This repository contains fine-tuned LoRA adapters for **Qwen2.5-{export_choice}-Instruct** trained on the FRED Stage 2 hybrid financial QA dataset.

## Operational Actions Supported
- **`preserve`**: The response is factually and numerically sound, completely grounded in the context.
- **`edit`**: Identifiable factual or numerical errors with verified in-place XML corrections (`<numerical>`, `<temporal>`, `<relation>`).
- **`abstain`**: Missing financial disclosures or ungrounded claims.
- **`flag_conflict`**: Contradictory statements between primary context and subsequent restatement footnotes.
- **`request_evidence`**: Irrelevant retrieved context requiring additional evidence.

## Training Details
- **Dataset**: 17,000 hybrid examples (8,500 clean + 8,500 controlled hallucination variants).
- **Data Sources**: FinQA (5,000 clean base) + TAT-QA (4,000 clean base).
- **Precision**: {'16-bit LoRA (Unquantized)' if export_choice == '3B' else '4-bit QLoRA'}
- **LoRA Parameters**: Rank 16, Alpha 16 (Targeting all attention and MLP projections).
- **Sequence Length**: 2,048 tokens.
- **Framework**: Unsloth + Hugging Face TRL.
"""
        with open("README.md", "w", encoding="utf-8") as f:
            f.write(model_card)
            
        api = HfApi(token=hf_token)
        api.upload_file(
            path_or_fileobj="README.md",
            path_in_repo="README.md",
            repo_id=hf_repo_id,
        )
        print(f"Model and Stage 2 README pushed successfully to https://huggingface.co/{hf_repo_id}!")
    else:
        print("Upload cancelled (no repo ID provided).")
else:
    print("Cannot push without a valid HF_TOKEN.")

--- 
# 7. Direct GGUF Export for Local Ollama Deployment
Export model directly to 4-bit GGUF format (`q4_k_m`) to download and run locally offline via Ollama.

In [ ]:
# Choose model to convert to GGUF
gguf_choice = input("Convert which model to GGUF? Enter '3B' or '7B': ").strip().upper()
target_model = model_7b if gguf_choice == "7B" else FastLanguageModel.from_pretrained("fred_stage2_qwen2_5_3b_lora")[0]
target_tok = tokenizer_7b if gguf_choice == "7B" else tokenizer_3b

gguf_output_name = f"fred-stage2-qwen-{gguf_choice.lower()}-q4_k_m"
print(f"Exporting {gguf_choice} model to {gguf_output_name}.gguf...")

target_model.save_pretrained_gguf(
    gguf_output_name,
    target_tok,
    quantization_method="q4_k_m"
)
print(f"GGUF file saved! Download {gguf_output_name}.gguf and deploy with Modelfile_stage2 in Ollama.")